# Global equity momentum and risk screen

Inspect Tyche's global listing coverage, then compare liquid country ETF proxies with transparent price-derived signals.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tyche_client import TycheClient

client = TycheClient(
    os.getenv("TYCHE_BASE_URL", "http://localhost:8000"),
    os.getenv("TYCHE_API_KEY", "tyk_paste_your_key_here"),
)

## Inspect cross-market coverage

Country codes come from Tyche's market-aware symbol classifier. Market caps remain in each listing's native currency, so this notebook does not compare or filter them across countries.

In [ ]:
market_proxies = pd.DataFrame({
    "country": ["US", "CA", "JP", "KR", "TW"],
    "ticker": ["SPY", "EWC", "EWJ", "EWY", "EWT"],
    "name": ["United States", "Canada", "Japan", "South Korea", "Taiwan"],
}).set_index("country")

coverage_sample = pd.concat(
    [client.universe(country=country, limit=3) for country in market_proxies.index],
    ignore_index=True,
)
coverage_sample[["ticker", "name", "country", "sector", "industry"]]

## Fetch prices and calculate comparable signals

The US-listed country ETFs avoid native-currency market-cap comparisons. Signals use only close prices returned by Tyche.

In [ ]:
prices = client.price_matrix(market_proxies["ticker"], start="2024-01-01").ffill()
valid = prices.columns[prices.notna().sum() >= 150]
prices = prices[valid].dropna()
if prices.shape[1] < 2:
    raise RuntimeError("At least two country ETFs need sufficient price history")
returns = prices.pct_change().dropna()

signals = pd.DataFrame({
    "momentum_3m": prices.pct_change(63).iloc[-1],
    "momentum_12m": prices.pct_change(252).iloc[-1],
    "volatility_3m": returns.tail(63).std() * np.sqrt(252),
    "drawdown": prices.iloc[-1] / prices.cummax().iloc[-1] - 1,
}).dropna()

signals = signals.join(market_proxies.reset_index().set_index("ticker")[["name", "country"]])
signals.head()

## Rank without hiding the model

Each metric is converted to a cross-sectional percentile. Higher momentum and shallower drawdowns help; higher volatility hurts.

In [ ]:
signals["score"] = (
    signals["momentum_3m"].rank(pct=True)
    + signals["momentum_12m"].rank(pct=True)
    + signals["drawdown"].rank(pct=True)
    + (1 - signals["volatility_3m"].rank(pct=True))
) / 4

columns = ["name", "country", "score", "momentum_3m", "momentum_12m", "volatility_3m", "drawdown"]
signals.sort_values("score", ascending=False)[columns].head(20).style.format({
    "score": "{:.2f}",
    "momentum_3m": "{:.1%}",
    "momentum_12m": "{:.1%}",
    "volatility_3m": "{:.1%}",
    "drawdown": "{:.1%}",
})

## Compare markets

The table and chart keep momentum and volatility visible alongside the composite rank.

In [ ]:
country_summary = signals.set_index("country")[[
    "score", "momentum_3m", "momentum_12m", "volatility_3m", "drawdown"
]].sort_values("score", ascending=False)
country_summary

In [ ]:
country_summary[["momentum_12m", "volatility_3m"]].plot.bar(
    figsize=(9, 5), title="Country ETF momentum and volatility"
)
plt.ylabel("annualized volatility / trailing return")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()

This screen is a reproducible research starting point, not an investment recommendation. Add transaction-cost and currency-hedging assumptions before using it as a strategy.